# 01. ETH staking yield와 ETH-BTC funding spread
가설1/2 검정


## Yield source note

기본 권장 yield는 `data/processed/eth_yield_panel.csv`의 Lido wstETH protocol exchange-rate 기반 `stake_yield`입니다.
이는 validator gross yield가 아니라 리테일 투자자가 접근 가능한 Lido LST yield proxy입니다.
stETH/ETH 또는 wstETH/ETH 시장가격 디스카운트/프리미엄은 yield가 아니라 별도 basis/depeg risk 변수로 다뤄야 합니다.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy import stats

from common.transforms import funding_to_daily_annualized, build_spread, log_return
from common.stats import ols_hac
from common.visualization import plot_timeseries


In [ ]:
RAW = Path('../../data/raw')
PROCESSED = Path('../../data/processed')

# funding 파일은 auto 수집 결과에 따라 binance_... 또는 bybit_... 일 수 있음
eth_f_path = next((p for p in [RAW/'binance_ethusdt_funding.csv', RAW/'bybit_ethusdt_funding.csv'] if p.exists()), None)
btc_f_path = next((p for p in [RAW/'binance_btcusdt_funding.csv', RAW/'bybit_btcusdt_funding.csv'] if p.exists()), None)

assert eth_f_path and btc_f_path, 'funding 파일이 없습니다. scripts/collect_binance_data.py 먼저 실행하세요.'

eth_f = pd.read_csv(eth_f_path)
btc_f = pd.read_csv(btc_f_path)
eth_p = pd.read_csv(RAW/'binance_ethusdt_1d.csv')
btc_p = pd.read_csv(RAW/'binance_btcusdt_1d.csv')

# staking yield 패널(권장): scripts/build_eth_yield_panel.py 결과물
st_path = PROCESSED/'eth_yield_panel.csv'
if st_path.exists():
    st = pd.read_csv(st_path)
else:
    # fallback: 직접 raw 파일 사용
    st_path = RAW/'eth_staking_yield_daily.csv'
    assert st_path.exists(), 'eth_yield_panel.csv 또는 eth_staking_yield_daily.csv 가 필요합니다.'
    st = pd.read_csv(st_path)


In [ ]:
# 1) funding -> daily annualized
eth_fd = funding_to_daily_annualized(eth_f, time_col='fundingTime', rate_col='fundingRate')
btc_fd = funding_to_daily_annualized(btc_f, time_col='fundingTime', rate_col='fundingRate')

# 2) spread 생성
spread = build_spread(eth_fd.rename(columns={'funding_ann':'funding_ann'}), btc_fd.rename(columns={'funding_ann':'funding_ann'}), value_col='funding_ann', out_col='spread')

# 3) 가격 수익률 생성
for px in (eth_p, btc_p):
    px['date'] = pd.to_datetime(px['date'])
eth_p['ret_eth'] = log_return(eth_p['close'])
btc_p['ret_btc'] = log_return(btc_p['close'])
ret = eth_p[['date','ret_eth']].merge(btc_p[['date','ret_btc']], on='date', how='inner')
ret['ret_eth_btc'] = ret['ret_eth'] - ret['ret_btc']

# 단순 RV proxy (절대수익률)
ret['rv_eth_btc'] = (ret['ret_eth'].abs() - ret['ret_btc'].abs())

# staking
st['date'] = pd.to_datetime(st['date'])
st['stake_yield'] = pd.to_numeric(st['stake_yield'], errors='coerce')
# stake_yield가 % 값이면 100으로 나누기 (예: 3.5 -> 0.035)
if st['stake_yield'].dropna().median() > 1:
    st['stake_yield'] = st['stake_yield'] / 100.0

spread['date'] = pd.to_datetime(spread['date'])
df = spread[['date','spread']].merge(ret[['date','ret_eth_btc','rv_eth_btc']], on='date', how='inner').merge(st[['date','stake_yield']], on='date', how='inner').dropna().sort_values('date').reset_index(drop=True)

df.head(), df.describe().T[['mean','std','min','max']]


In [ ]:
# 기초 시계열 시각화
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
plot_timeseries(df, 'date', 'spread', 'Funding Spread (ETH - BTC, annualized)', ax=axes[0])
plot_timeseries(df, 'date', 'stake_yield', 'ETH Staking Yield (annualized)', ax=axes[1])
plot_timeseries(df, 'date', 'ret_eth_btc', 'ETH/BTC Relative Return', ax=axes[2])
plt.tight_layout()


In [ ]:
# 상관관계
corr = df[['spread','stake_yield','ret_eth_btc','rv_eth_btc']].corr()
corr


In [ ]:
# 가설1: 기본 회귀 spread ~ stake_yield
res1 = ols_hac(df['spread'], df[['stake_yield']], maxlags=5)
print(res1.summary())


In [ ]:
# 가설2: 통제회귀 spread ~ stake_yield + ret_eth_btc + rv_eth_btc
res2 = ols_hac(df['spread'], df[['stake_yield','ret_eth_btc','rv_eth_btc']], maxlags=5)
print(res2.summary())


In [ ]:
# High/Low staking yield 그룹 평균 차이 t-test
median_y = df['stake_yield'].median()
high = df.loc[df['stake_yield'] > median_y, 'spread']
low = df.loc[df['stake_yield'] <= median_y, 'spread']
t_stat, p_val = stats.ttest_ind(high, low, equal_var=False, nan_policy='omit')
print({'high_mean': high.mean(), 'low_mean': low.mean(), 'diff': high.mean()-low.mean(), 't': t_stat, 'p': p_val})
